# Mar 10/11 · 15:45 — TP v0.9 : HTTP et GET /health

Mar 10/11, TP de 16:15 à 17:10. Correction à 17:10, TP fini ou non. La référence v0.9 est publiée à 17:15.

## Objectif

Le ping dit qu'une machine répond, pas que le service tourne. `fleetcheck` interroge maintenant chaque service par HTTP :

- `GET /health` avec `requests`, toujours avec un délai maximal ;
- les erreurs réseau (connexion refusée, délai dépassé) sont rattrapées et deviennent une raison, pas un plantage ;
- un code hors `2xx` rend le service `down`, une réponse lente le rend `degraded`.

## Mode d'emploi du notebook

- Copiez ce notebook à la racine de votre dépôt, à côté de `fleetcheck.py`. Le code s'écrit dans `fleetcheck.py`, ouvert dans un onglet de JupyterLab ; le notebook sert à essayer et à vérifier chaque étape.
- **Le serveur simulé doit tourner pendant tout le TP.** Dans un terminal de JupyterLab, venv activé, depuis votre clone du dépôt public : `python simserver.py`. Laissez ce terminal ouvert jusqu'à 19:00 : il affiche une ligne par requête reçue.
- Port `8000` déjà pris : `python simserver.py --base-port 9000`. Écrivez l'inventaire correspondant à côté de `fleetcheck.py` : `python simserver.py --base-port 9000 --inventory > servers-sim.csv` (sous Windows PowerShell : `python simserver.py --base-port 9000 --inventory | Set-Content -Encoding ascii servers-sim.csv`, car `>` y écrit de l'UTF-16 que `load()` ne lit pas). Puis `BASE = 9000` dans la cellule prévue plus bas, et `"--inventory", "servers-sim.csv"` ajouté à chaque `run()`.
- Exécutez d'abord la cellule d'outils ci-dessous. Chaque vérification affiche `OK`, ou `KO` avec ce qui était attendu.
- `%run -n fleetcheck.py` charge les fonctions du fichier enregistré sans exécuter le bloc `if __name__ == "__main__":`. N'appelez pas `main()` dans le notebook.
- `run(...)` lance le script comme dans un terminal et renvoie `r.stdout`, `r.stderr` et `r.returncode`. Un run complet dure environ 5 s : latence de `web-2`, délai de `api-stg-1`, ping de `db-1`.
- `worker-1` répond `503` ou `200` selon le moment de son cycle : aucune vérification ne teste son statut.
- **Prédire** et **Justifier** marquent les moments clés du TP. Répondez dans la cellule `# Ma réponse :` qui suit, **avant** d'exécuter la suite.
- Rendu en fin de TP : menu *Kernel > Restart Kernel and Run All Cells*, serveur simulé lancé, puis commit et push des mêmes fichiers qu'en v0.8 et de ce notebook, sans `reports/`.

In [ ]:
# Outils du TP : exécuter cette cellule en premier, sans la modifier.
import os
import subprocess
import sys


def check(obtenu, attendu):
    if obtenu == attendu:
        print("OK")
    elif isinstance(attendu, str) and "\n" in attendu:
        print(f"KO, obtenu :\n{obtenu}\nattendu :\n{attendu}")
    else:
        print(f"KO\n  obtenu  : {obtenu!r}\n  attendu : {attendu!r}")


def check_error(fonction, *args, attendu):
    try:
        fonction(*args)
    except Exception as e:
        check(f"{type(e).__name__}: {e}", attendu)
    else:
        print(f"KO, aucune exception levée. Attendu : {attendu}")


def run(*args):
    """Lance fleetcheck.py comme dans un terminal (vu en v0.8 : subprocess)."""
    r = subprocess.run([sys.executable, "fleetcheck.py", *args], capture_output=True,
                       encoding="utf-8", env={**os.environ, "PYTHONUTF8": "1", "NO_COLOR": "1",
                                              "PYTHONWARNDEFAULTENCODING": "1"})
    print(r.stdout, end="")
    print(r.stderr, end="", file=sys.stderr)
    print(f"[code de sortie : {r.returncode}]")
    if "EncodingWarning" in r.stderr:
        print("KO : un open(), read_text() ou subprocess text=True sans encoding= (ligne indiquée dans l'EncodingWarning ci-dessus)")
    return r

Port du serveur simulé : `8000` par défaut. Si vous l'avez lancé avec `--base-port 9000`, mettez `BASE = 9000` ici avant d'exécuter la suite : toutes les URL des cellules en dépendent.

In [ ]:
BASE = 8000

## Point de départ

Votre `fleetcheck.py` de v0.8, avec `config.json`, `requirements.txt`, `.gitignore` et les deux inventaires.

En retard ou en panne : copiez dans votre dépôt les fichiers de `reference/` au tag `v0.8` du dépôt public.

Le scénario du serveur simulé, un comportement par service :

| Service | Port | Comportement de `/health` |
|---|---|---|
| `web-1`, `api-1`, `web-stg-1` | 8001, 8003, 8006 | `200` immédiat |
| `web-2` | 8002 | `200` après 1,5 s |
| `api-2` | 8004 | `503` : le processus tourne, sa base est tombée |
| `worker-1` | 8005 | Cycle `503`, `503`, `200` |
| `api-stg-1` | 8007 | Répond au bout de 10 s |
| `cache-1` | 8009 | Rien n'écoute : connexion refusée |
| `db-1` | — | `192.0.2.1`, ping KO |

Vérifiez avant de commencer :

In [ ]:
import json
import time
from pathlib import Path

%run -n fleetcheck.py
check((callable(ping), callable(check_service), callable(global_status)), (True, True, True))
config = json.loads(Path("config.json").read_text(encoding="utf-8"))
check((config["timeout_s"], config["degraded_s"]), (2, 1.0))

## Étapes

| Fin | Étape |
|---|---|
| 16:30 | 1. Serveur simulé lancé, premiers `requests.get` |
| 16:50 | 2. `check_health(url, timeout_s)` |
| 17:10 | 3. `check_service` applique la règle de statut ; sortie et rapport |

À 16:40, si plus d'un tiers de la salle n'a pas de `200` à l'étape 1, l'enseignant passe dans les rangs ; le cas `8007` de l'étape 2 est montré à la correction.

### 1. Premiers appels — fin 16:30

Serveur simulé lancé (mode d'emploi). Dans le notebook :

- `requests.get("http://127.0.0.1:8001/health", timeout=2)` ;
- lisez `r.status_code`, `r.json()` et `r.elapsed` ;
- recommencez sur `8004`.

**Prédire** — `requests.get("http://127.0.0.1:8004/health", timeout=2)` lève-t-il une exception ?

In [ ]:
# Ma réponse :

Vérification :

In [ ]:
import requests

r = requests.get(f"http://127.0.0.1:{BASE + 1}/health", timeout=2)
check((r.status_code, r.json()), (200, {"service": "web-1", "status": "ok"}))
r = requests.get(f"http://127.0.0.1:{BASE + 4}/health", timeout=2)
check(r.status_code, 503)

### 2. `check_health` — fin 16:50

`check_health(url, timeout_s)` fait un seul `GET` et ne lève jamais d'exception. Elle renvoie un dict de 4 clés :

| Clé | Réponse reçue | `requests.Timeout` | `requests.ConnectionError` |
|---|---|---|---|
| `http_status` | `response.status_code` | `None` | `None` |
| `latency_s` | `round(response.elapsed.total_seconds(), 3)` | `None` | `None` |
| `attempts` | `1` | `1` | `1` |
| `error` | `None` | `"timeout"` | `"connexion impossible"` |

`attempts` vaut toujours `1` en v0.9 ; il comptera les essais en v0.10. `import requests` en tête de `fleetcheck.py` : le paquet est installé et épinglé depuis v0.6.

Vérification : un service sain, un `503`, un port fermé, un délai dépassé (2 s d'attente), un service lent.

In [ ]:
%run -n fleetcheck.py
sain = check_health(f"http://127.0.0.1:{BASE + 1}/health", 2)
check((sain["http_status"], sain["attempts"], sain["error"]), (200, 1, None))
check(check_health(f"http://127.0.0.1:{BASE + 4}/health", 2)["http_status"], 503)
check(check_health(f"http://127.0.0.1:{BASE + 9}/health", 2),
      {"http_status": None, "latency_s": None, "attempts": 1, "error": "connexion impossible"})
debut = time.monotonic()
lent = check_health(f"http://127.0.0.1:{BASE + 7}/health", 2)
check((lent["error"], time.monotonic() - debut < 3), ("timeout", True))
check(check_health(f"http://127.0.0.1:{BASE + 2}/health", 2)["latency_s"] > 1.0, True)

### 3. Statut d'un service — fin 17:10

Dans `check_service`, après un ping réussi seulement, appelez `check_health` sur `http://<ip>:<port>/health` avec `config["timeout_s"]`, et rangez le résultat sous la clé `health` (`None` si le ping a échoué). Règle, dans l'ordre :

| Condition | Statut | Raison |
|---|---|---|
| Ping KO | `down` | `ping KO` (on s'arrête là) |
| `error` non nul | `down` | la valeur de `error` : `timeout` ou `connexion impossible` |
| code hors `200 <= code < 300` | `down` | `HTTP <code>`, par exemple `HTTP 503` |
| `latency_s > config["degraded_s"]` | `degraded` | `latence 1.50 s > 1.0 s` (`f"latence {latence:.2f} s > {seuil} s"`) |
| sinon | `ok` | — |

`main()`, `global_status`, `exit_code` et `as_text` ne changent pas : `degraded` sort en `0`.

**Justifier** — Pourquoi tester le ping avant `/health`, et s'arrêter s'il échoue ?

In [ ]:
# Ma réponse :

Vérification, `--env staging` : environ 3 s.

In [ ]:
r = run("--env", "staging")
check((r.stdout, r.returncode), ("""ok        web-stg-1    staging
down      api-stg-1    staging  timeout

DOWN : 1 ok, 0 degraded, 1 down, 0 invalide(s)
""", 1))

Vérification, inventaire complet en JSON : statut de chaque service, sauf `worker-1`.

In [ ]:
r = run("--format", "json")
services = {s["name"]: s for s in json.loads(r.stdout)["services"]}
statuts = {nom: s["status"] for nom, s in services.items() if nom != "worker-1"}
check(statuts, {"web-1": "ok", "web-2": "degraded", "api-1": "ok", "api-2": "down",
                "web-stg-1": "ok", "api-stg-1": "down", "cache-1": "down", "db-1": "down"})
check((services["api-2"]["reasons"], services["cache-1"]["reasons"], services["db-1"]["health"]),
      (["HTTP 503"], ["connexion impossible"], None))
check(services["web-2"]["reasons"][0].startswith("latence 1."), True)
check(r.returncode, 1)

**Prédire** — Vous relancez deux fois de suite `run()` sans redémarrer le serveur. `worker-1` a-t-il le même statut les deux fois ?

In [ ]:
# Ma réponse :

In [ ]:
for _ in range(2):
    r = run("--format", "json")
    print("worker-1 :", next(s["status"] for s in json.loads(r.stdout)["services"] if s["name"] == "worker-1"))

## Critère final

| Service | Statut | Raison |
|---|---|---|
| `web-1`, `api-1`, `web-stg-1` | `ok` | — |
| `web-2` | `degraded` | `latence 1.5x s > 1.0 s` |
| `api-2` | `down` | `HTTP 503` |
| `worker-1` | `down` ou `ok` | selon son cycle |
| `api-stg-1` | `down` | `timeout` |
| `cache-1` | `down` | `connexion impossible` |
| `db-1` | `down` | `ping KO` |

Code de sortie `1`. Ce sont les vérifications de l'étape 3.

## Pour aller plus loin

- `curl -i http://127.0.0.1:8004/health` dans un terminal : retrouvez le code, l'en-tête `Content-Type` et le corps JSON que `requests` vous a rendus.
- Le corps de `/health` de `api-2` dit pourquoi il est en panne (`reason`). Ajoutez-le à la raison du service quand il existe, sans casser le cas où le corps n'est pas du JSON (`requests.JSONDecodeError`).